# Lab 9.11 &mdash; Assemble the Insight Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Assemble read-only tools + model + prompt into an executor
- Withhold any trade/advice tool so it cannot act
- Return a grounded, cited insight flagged needs_review

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Modules 6&ndash;8. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; The financial-report insight agent, end to end](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-09-11"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Now assemble the insight agent from Modules 6&ndash;8 pieces (deck slides 7&ndash;9): `@tool`
**read-only** tools (`extract_figure`, `compute`), `create_react_agent`, an `AgentExecutor`. The
scripted model grounds a figure, computes a metric, and drafts a **cited** Final Answer. The design
choice: the tools list is **read-only** &mdash; no `place_trade`, no `give_advice` &mdash; and the
output is flagged **`needs_review`** for a human analyst.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

def create_react_agent(model, tools, prompt):
    """Bind model + tools + prompt into a ReAct agent (mirrors langchain.agents.create_react_agent)."""
    return {"model": model, "tools": {t.name: t for t in tools}, "prompt": prompt}
def parse_react(text):
    """Turn the model's ReAct text into ('final', answer) or ('action', name, input)."""
    action = inp = None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("Final Answer:"): return ("final", s.split(":", 1)[1].strip())
        if s.startswith("Action Input:"): inp = s.split(":", 1)[1].strip()
        elif s.startswith("Action:"):      action = s.split(":", 1)[1].strip()
    return ("action", action, inp)
class AgentExecutor:
    """Runs the loop: ask model -> parse -> run tool -> observe -> repeat, capped by max_iterations
    (mirrors langchain.agents.AgentExecutor). verbose=True prints the ReAct trace."""
    def __init__(self, agent, tools=None, verbose=False, max_iterations=8):
        self.agent = agent
        self.tools = agent["tools"] if tools is None else {t.name: t for t in tools}
        self.model = agent["model"]; self.prompt = agent["prompt"]
        self.verbose = verbose; self.max_iterations = max_iterations
    def invoke(self, inputs):
        scratch, steps = "", []
        for _ in range(self.max_iterations):
            text = self.model.invoke(self.prompt.format(input=inputs["input"], scratchpad=scratch)).content
            if self.verbose: print(text)
            parsed = parse_react(text)
            if parsed[0] == "final":
                return {"output": parsed[1], "intermediate_steps": steps}
            name, arg = parsed[1], parsed[2]
            obs = self.tools[name].invoke(arg) if name in self.tools else ("unknown tool: %s" % name)
            if self.verbose: print("Observation:", obs)
            steps.append((name, arg, obs)); scratch += text + "\nObservation: " + str(obs) + "\n"
        return {"output": None, "intermediate_steps": steps}

# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

@tool
def extract_figure(name):
    """Pull a reported figure and its source from the filing."""
    return REPORT.get(name, {})
@tool
def compute(expression):
    """Do exact arithmetic on a financial expression."""
    return safe_calc(expression)
print("read-only tools ready:", extract_figure.name, "&", compute.name)

## Your Turn
Complete `make_insight_agent` (read-only tools + step cap) and `analyze_report` (flag needs_review).

In [ ]:
SCRIPT = [
    "Thought: get revenue, grounded.\nAction: extract_figure\nAction Input: revenue",
    "Thought: compute YoY vs 107.1.\nAction: compute\nAction Input: (120-107.1)/107.1*100",
    "Thought: draft a cited insight.\nFinal Answer: Revenue 120.0M [p4, income stmt], +12.0% YoY.",
]

def make_insight_agent(max_iterations=8):
    model  = FakeChatModel(SCRIPT)
    prompt = PromptTemplate.from_template("Report: {input}\n{scratchpad}")
    tools  = ___   # TODO: read-only -- [extract_figure, compute], NO trade/advice tool
    agent  = create_react_agent(model, tools, prompt)
    return AgentExecutor(agent, max_iterations=___)   # TODO: the step cap

def analyze_report(report_name):
    result = make_insight_agent().invoke({"input": report_name})
    # analysis only: flag for a human analyst, never a decision
    return {"insight": result["output"], "status": ___,   # TODO: needs_review
            "tools_used": [s[0] for s in result["intermediate_steps"]]}

try:
    out = analyze_report('ACME Q3')
    print('tools used:', out['tools_used'])
    print('status    :', out['status'])
    print('insight   :', out['insight'])
    print('has trade tool?', 'place_trade' in [t.name for t in make_insight_agent().tools.values()])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the agent grounds via extract_figure first", lambda: analyze_report("x")["tools_used"][0] == "extract_figure")
expect_true("it then computes a metric", lambda: "compute" in analyze_report("x")["tools_used"])
expect_true("the insight cites its source", lambda: "p4" in analyze_report("x")["insight"])
expect_true("the output is needs_review, never a decision", lambda: analyze_report("x")["status"] == "needs_review")
expect_true("the agent holds NO trade tool", lambda: "place_trade" not in [t.name for t in make_insight_agent().tools.values()])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain (not graded)
Swap the scripted model for a REAL LangChain agent (Ollama llama3.2:1b, or Groq) -- read-only, grounded. Safe to skip &mdash; it needs `pip install langchain langchain-ollama` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`. If a package, model or key is
missing the cell prints a friendly note and moves on.
**The graded steps above are offline &amp; deterministic, so the lab always verifies without a model.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    out = llm.invoke("Summarize in one line, cite the page: revenue 120M on p4, up 12% YoY. Give NO advice.").content
    print("REAL insight draft:", out)
    print("In production: bind extract_figure/compute (read-only) with create_react_agent + AgentExecutor,")
    print("and simply DON'T include a trade or advice tool -- the agent can analyse but cannot act.")
except Exception as e:
    print("No local LLM available -- skipping (pip install langchain langchain-ollama + `ollama run llama3.2:1b`,")
    print("or langchain-groq with GROQ_API_KEY):", type(e).__name__)
    print("The offline agent above already produced a grounded, cited, needs_review insight.")

---
### Key takeaway
Same executor as Module 6, pointed at a filing -- and the guardrail is what's MISSING from the tools list. The agent grounds, computes and cites; it cannot trade or advise. Next: run it over a suite.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>